# Módulo 4: Análise Exploratória e Regressão Linear Simples

> **Resumo Didático Integrado** — Baseado no *Capítulo 3 (Análise de Dados — Seção 1: Regressão Linear Simples)*.
> 
> Este caderno é **100% autocontido**: você aprenderá a formular, ajustar, interpretar e validar estatisticamente um modelo de **Regressão Linear Simples** em R, compreendendo o método dos Mínimos Quadrados, a interpretação rigorosa do `summary()` e o cálculo das métricas de erro (**MAE**, **MSE** e **RMSE**).

---

## 1. Visão Geral e Fundamentação Matemática

A **Regressão Linear** é um dos pilares da Estatística e do Aprendizado de Máquina Supervisionado. Seu objetivo é quantificar a relação entre uma variável de resposta contínua ($Y$, dependente) e uma variável explicativa ($X$, independente ou preditora).

### 1.1 A Equação da Reta
A relação populacional teórica é expressa como:

$$Y = \beta_0 + \beta_1 X + \epsilon$$

Onde:
* $\beta_0$ (**Intercepto**): O valor esperado de $Y$ quando $X = 0$.
* $\beta_1$ (**Coeficiente Angular / Inclinação**): O acréscimo (ou decréscimo) médio esperado em $Y$ para cada unidade adicional em $X$.
* $\epsilon$ (**Termo de Erro Aleatório / Resíduo**): Captura as variações de $Y$ não explicadas pela relação linear com $X$.

### 1.2 O Método dos Mínimos Quadrados Ordinários (MQO / OLS)
O R calcula os coeficientes amostrais $b_0$ e $b_1$ minimizando a **Soma dos Quadrados dos Resíduos (SQRes)**:

$$\min \sum_{i=1}^{n} e_i^2 = \min \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Onde $\hat{y}_i = b_0 + b_1 x_i$ é o valor predito pelo modelo para a observação $i$.

---

## 2. Indicadores de Qualidade do Ajuste

### 2.1 Coeficiente de Determinação ($R^2$)
O $R^2$ varia entre $0$ e $1$ (ou $0\%$ a $100\%$) e mede a proporção da variabilidade total da variável $Y$ que é explicada pela variável preditora $X$:

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

* Um $R^2 = 0.85$ significa que $85\%$ da flutuação observada em $Y$ é justificada linearmente pelas variações de $X$.

### 2.2 Métricas de Erro Preditivo
Para avaliar a precisão das predições do modelo na prática:
* **EAM / MAE (*Mean Absolute Error*)**:
  $$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$
  Representa a média dos desvios absolutos, expressa exatamente na mesma unidade de medida de $Y$.
* **EQM / MSE (*Mean Squared Error*)**:
  $$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$
  Eleva os desvios ao quadrado, penalizando severamente modelos que cometem grandes erros isolados.
* **REQM / RMSE (*Root Mean Squared Error*)**:
  $$RMSE = \sqrt{MSE}$$
  Traz o erro quadrático de volta para a escala original de $Y$, equivalendo ao desvio padrão dos resíduos.

---

## 3. Passo a Passo Prático: Modelagem do Início ao Fim

Seguiremos a tríade didática:
* 🎯 **O que fazer**
* 💻 **Como fazer**
* 💡 **Por que fazer**

### Passo 1: Preparação dos Dados e Análise Exploratória Bivariada

🎯 **O que fazer**:
Limpar a memória, gerar/carregar um conjunto de dados imobiliário (relação entre área construída em $m^2$ e preço de venda), calcular o coeficiente de correlação linear de Pearson e visualizar o diagrama de dispersão.

💻 **Como fazer**:

In [ ]:
# Limpeza de ambiente
rm(list = ls())

# Configuração gráfica
if (requireNamespace("repr", quietly = TRUE)) {
  options(repr.plot.width = 11, repr.plot.height = 5)
}

# Simulação controlada e reproduzível baseada em dados reais de mercado
set.seed(123)
n_amostras <- 100
area_m2 <- round(runif(n_amostras, min = 45, max = 280), 1)

# Preço com relação linear subjacente (beta0 = 80 mil, beta1 = 4.5 mil/m² + ruído aleatório)
preco_mil <- 80 + 4.5 * area_m2 + rnorm(n_amostras, mean = 0, sd = 75)

dados_imoveis <- data.frame(area = area_m2, preco = preco_mil)

# 1. Cálculo da Correlação Linear de Pearson (r)
r_pearson <- cor(dados_imoveis$area, dados_imoveis$preco)
cat("Coeficiente de Correlação de Pearson (r):", round(r_pearson, 4), "\n")

# 2. Visualização gráfica preliminar (Scatterplot)
plot(dados_imoveis$area, dados_imoveis$preco,
     main = paste("Dispersão: Preço vs Área (Correlação r =", round(r_pearson, 2), ")"),
     xlab = "Área Construída (m²)",
     ylab = "Preço de Venda (R$ mil)",
     pch = 21, bg = "#3a86ff", col = "#1d3557", cex = 1.2)
grid()

💡 **Por que fazer**:
A correlação de Pearson ($r$) deve ser calculada antes do modelo para confirmar se existe uma tendência linear estatisticamente consistente entre as duas variáveis ($r$ próximo de $+1$ ou $-1$). Se o gráfico de dispersão revelar uma parábola ou curva exponencial, a regressão linear simples pura será ineficaz sem transformações prévias.

### Passo 2: Ajuste do Modelo com a Função `lm()`

🎯 **O que fazer**:
Ajustar o modelo linear utilizando a função nativa `lm(formula, data)` do R, onde a variável resposta fica à esquerda do til (`~`) e a variável preditora à direita.

💻 **Como fazer**:

In [ ]:
# Ajuste do modelo de Regressão Linear Simples: preco ~ area
modelo_linear <- lm(preco ~ area, data = dados_imoveis)

# Exibição dos coeficientes fundamentais
coeficientes <- coef(modelo_linear)
b0 <- coeficientes[1] # Intercepto
b1 <- coeficientes[2] # Coeficiente Angular

cat("Equação estimada da reta:\n")
cat("Preço Estimado = ", round(b0, 2), " + ", round(b1, 2), " * Área\n", sep = "")

💡 **Por que fazer**:
A função `lm()` executa a decomposição matricial QR para calcular as estimativas $b_0$ e $b_1$ com máxima estabilidade numérica. A equação obtida permite transformar hipóteses teóricas em uma fórmula preditiva direta e operacionalizável.

### Passo 3: Interpretação Completa e Rigorosa do `summary()`

🎯 **O que fazer**:
Analisar as quatro seções do relatório `summary()`: resíduos, significância estatística dos coeficientes ($p$-valor), erro padrão residual e coeficiente de determinação ($R^2$).

💻 **Como fazer**:

In [ ]:
resumo <- summary(modelo_linear)
print(resumo)

# Extraindo e interpretando os valores chave programmaticamente:
p_valor_area <- resumo$coefficients["area", "Pr(>|t|)"]
r2 <- resumo$r.squared
r2_ajustado <- resumo$adj.r.squared

cat("\n--- Diagnóstico Estatístico Detalhado ---\n")
cat("1. P-valor da Área:", format.pval(p_valor_area, digits = 3), "\n")
if (p_valor_area < 0.05) {
  cat("   -> O coeficiente é estatisticamente significante (rejeita-se H0: beta1 = 0 com p < 0.05).\n")
}

cat("2. Coeficiente de Determinação (R²):", round(r2 * 100, 2), "%\n")
cat("   ->", round(r2 * 100, 2), "% da variação dos preços é explicada unicamente pela área construída.\n")

cat("3. Interpretação Prática do Coeficiente Angular (beta1 = ", round(b1, 2), "):\n", sep = "")
cat("   -> Cada metro quadrado adicional construído valoriza o imóvel, em média, em R$", round(b1 * 1000, 2), ".\n")

💡 **Por que fazer**:
Um modelo pode apresentar coeficientes altos por puro acaso amostral. Verificar se o $p$-valor associado à estatística $t$ é inferior a $0.05$ (ou $0.01$) garante que a relação observada não é ruído estatístico, legitimando a tomada de decisão corporativa.

### Passo 4: Cálculo Manual e Validação das Métricas de Erro (MAE, MSE, RMSE)

🎯 **O que fazer**:
Obter as predições $\hat{y}$ do modelo, calcular os resíduos $e_i = y_i - \hat{y}_i$ e computar formalmente as métricas de erro preditivo (**MAE**, **MSE** e **RMSE**).

💻 **Como fazer**:

In [ ]:
# 1. Valores preditos pelo modelo ajustado
y_predito <- fitted(modelo_linear)

# 2. Resíduos (diferença entre valor real observado e valor predito)
residuos <- dados_imoveis$preco - y_predito

# 3. Funções utilitárias de avaliação de erro
calc_mae <- function(real, pred) {
  return(mean(abs(real - pred)))
}

calc_mse <- function(real, pred) {
  return(mean((real - pred)^2))
}

calc_rmse <- function(real, pred) {
  return(sqrt(calc_mse(real, pred)))
}

mae_valor  <- calc_mae(dados_imoveis$preco, y_predito)
mse_valor  <- calc_mse(dados_imoveis$preco, y_predito)
rmse_valor <- calc_rmse(dados_imoveis$preco, y_predito)

cat("--- Métricas de Erro Preditivo ---\n")
cat("MAE  (Erro Absoluto Médio):", round(mae_valor, 2), "mil reais (erro médio esperado por imóvel)\n")
cat("MSE  (Erro Quadrático Médio):", round(mse_valor, 2), "\n")
cat("RMSE (Raiz do Erro Quadrático):", round(rmse_valor, 2), "mil reais (desvio padrão do erro)\n")

💡 **Por que fazer**:
O $R^2$ informa quão bem o modelo explica a variância, mas não expressa o erro em termos financeiros. O MAE e o RMSE colocam o erro na mesma moeda do negócio (milhares de reais), permitindo que diretores e gestores avaliem se a margem de incerteza do modelo é aceitável para precificação.

### Passo 5: Visualização da Reta Ajustada e Diagnóstico de Resíduos

🎯 **O que fazer**:
Plotar a reta de regressão ajustada sobre a nuvem de pontos com `abline()` e inspecionar a distribuição dos resíduos para verificar os pressupostos estatísticos.

💻 **Como fazer**:

In [ ]:
par(mfrow = c(1, 2))

# 1. Reta ajustada sobre os dados
plot(dados_imoveis$area, dados_imoveis$preco,
     main = "Reta de Regressão Estimada",
     xlab = "Área (m²)", ylab = "Preço (R$ mil)",
     pch = 19, col = "#457b9d")
abline(modelo_linear, col = "#e63946", lwd = 2.5)
legend("topleft", legend = c("Dados Reais", "Reta MQO"),
       col = c("#457b9d", "#e63946"), pch = c(19, NA), lty = c(NA, 1), lwd = c(NA, 2.5))
grid()

# 2. Gráfico de Resíduos vs Valores Ajustados (Diagnóstico de Homocedasticidade)
plot(y_predito, residuos,
     main = "Diagnóstico: Resíduos vs Ajustados",
     xlab = "Preço Predito (R$ mil)", ylab = "Resíduo (Real - Predito)",
     pch = 19, col = "#2a9d8f")
abline(h = 0, col = "red", lty = 2, lwd = 2) # Linha zero ideal
grid()

par(mfrow = c(1, 1))

💡 **Por que fazer**:
O gráfico de resíduos vs. valores ajustados é o teste visual de **homocedasticidade** e **linearidade**. Se os resíduos estiverem dispersos aleatoriamente em torno de zero, sem formar funis ou curvas (efeito "leque"), os pressupostos do modelo clássico são válidos e as predições são confiáveis.

## 4. Conclusão e Próximos Passos
Neste Módulo 4, consolidamos:
* A formulação matemática da **Regressão Linear Simples**.
* O princípio de otimização dos **Mínimos Quadrados Ordinários (MQO)**.
* O significado prático de cada linha do relatório `summary()` (coeficientes, $p$-valor e $R^2$).
* O cálculo e interpretação das métricas de erro **MAE, MSE e RMSE**.
* A validação visual do modelo e o diagnóstico básico de resíduos.

Contudo, fenômenos reais dependem de múltiplas variáveis concorrentes. No **Módulo 5 (`05_Regressao_Linear_Multipla_e_Padronizacao.ipynb`)**, expandiremos a regressão para múltiplos preditores simultâneos, introduzindo termos polinomiais, diagnóstico de multicolinearidade (VIF) e o estudo aprofundado de **Padronização de Variáveis**.